<a href="https://colab.research.google.com/github/1833050309/IB9AU/blob/main/task15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU scikit-learn
!pip install -qU transformers datasets


In [ ]:
from llama_index.core.schema import Document

# Initialize an empty list to store the generated Document objects
qa_documents = []

# Iterate through each row of the df DataFrame
for index, row in df.iterrows():
    # Concatenate relevant column content into a single text field
    document_text = (
        f"Complaint: {row['complaint']}\n"
        f"Relevant Policy: {row['relevant_policy']}\n"
        f"Resolution: {row['resolution']}\n"
        f"Validity: {row['validity']}"
    )

    # Create a Document object
    doc = Document(
        text=document_text,
        metadata={
            "policy_category": row['policy_category']
        }
    )

    # Append the created Document object to the list
    qa_documents.append(doc)

# Print the total number of Document objects created
print(f"\u2705 Total Document objects created: {len(qa_documents)}")

# Display the content of the first Document to verify its structure
print("\n\u2705 Content of the first Document (qa_documents[0]):")
print(qa_documents[0].text)
print("\nMetadata of the first Document:")
print(qa_documents[0].metadata)

In [ ]:
from llama_index.core import VectorStoreIndex

print("\U0001f9e0 Building Text Vector Index from qa_documents...")
text_vector_index = VectorStoreIndex.from_documents(qa_documents)
print("\u2705 Text Vector Index built successfully!")

In [ ]:
print("⌛ Creating Text Query Engine...")
text_query_engine = text_vector_index.as_query_engine()
print("✅ Text Query Engine created successfully!")

In [ ]:
print("\U0001f50e Testing Text RAG System with example queries...")

# Define a list of relevant questions
questions = [
    "What is the policy for 4% cashback purchases at grocery stores?",
    "Explain the conditions for Purchase Security for damaged items.",
    "What are the rules for recurring bill payments cashback?",
    "What happens if a merchant sells both eligible and non-eligible items for cashback?"
]

# Iterate through each question and query the text_query_engine
for i, q in enumerate(questions):
    print(f"\n--- Query {i+1} ---")
    print(f"Question: {q}")
    response = text_query_engine.query(q)
    print(f"Answer: {response}")

print("\n\u2705 Text RAG System testing complete.")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 确认 df 已经加载，如果尚未加载，则重新加载
if 'df' not in locals():
    df = pd.read_csv('credit_card_qa.csv')
    print("DataFrame 'df' loaded successfully from 'credit_card_qa.csv'.")
else:
    print("DataFrame 'df' already exists in the environment.")

# 编码 policy_category
label_encoder_policy = LabelEncoder()
df['policy_category_encoded'] = label_encoder_policy.fit_transform(df['policy_category'])
print(f"\nEncoded 'policy_category' labels: {label_encoder_policy.classes_}")

# 编码 resolution
label_encoder_resolution = LabelEncoder()
df['resolution_encoded'] = label_encoder_resolution.fit_transform(df['resolution'])
print(f"Encoded 'resolution' labels: {label_encoder_resolution.classes_}")

# 准备数据集用于微调
# 任务要求：取 60 条记录进行微调，并在所有记录上评估。
# 我们将把整个数据集分为训练集和评估集。训练集将包含 60 条记录。

# 确保数据量足够，如果不足 60 条，则使用所有数据进行训练
if len(df) < 60:
    train_df = df
    eval_df = df
    print(f"Dataset has {len(df)} records, using all for training and evaluation.")
else:
    # 分割训练集 (60条) 和剩余数据作为评估集
    train_df, eval_df = train_test_split(df, train_size=60, stratify=df['policy_category'], random_state=42)
    print(f"Total records: {len(df)}")
    print(f"Training records: {len(train_df)}")
    print(f"Evaluation records: {len(eval_df)}")

print("\nDataFrame head with encoded labels:")
display(df.head())


## Summary:

### Q&A
The new text RAG system built for `credit_card_qa.csv` effectively retrieves relevant contextual information based on user queries, as demonstrated by the test cases. Its capabilities currently focus on identifying and surfacing pertinent data points from the credit card FAQ dataset.

To further integrate a real LLM and enhance text generation capabilities, the `Settings.llm` configuration within the LlamaIndex framework needs to be updated from `None` (which defaults to `MockLLM`) to a concrete LLM instance (e.g., `OpenAI()`, `LlamaCPP()`, etc.). Once a real LLM is configured, the `text_query_engine` will be able to synthesize more sophisticated and human-like answers by processing the retrieved context through the LLM, rather than just returning the raw context as the `MockLLM` does.

### Data Analysis Key Findings
*   The `credit_card_qa.csv` file, containing 80 rows of credit card-related inquiries and policies, was successfully loaded into a Pandas DataFrame.
*   Eighty LlamaIndex `Document` objects were created from the DataFrame. Each document's `text` field combined `complaint`, `relevant_policy`, `resolution`, and `validity` data, while `policy_category` was stored as metadata.
*   A text vector index (`text_vector_index`) was successfully built from these `Document` objects, enabling semantic search capabilities.
*   A query engine (`text_query_engine`) was successfully created from the vector index. It was noted that the system was configured to use `MockLLM` for response generation, as no specific LLM was defined (`Settings.llm` was `None`).
*   Testing of the RAG system confirmed its ability to retrieve relevant contextual information for various questions, such as "What is the policy for 4% cashback purchases at grocery stores?" and "Explain the conditions for Purchase Security for damaged items."
*   Due to the use of `MockLLM`, the answers provided by the query engine primarily consisted of the retrieved context, confirming that the retrieval mechanism functions as expected.

### Insights or Next Steps
*   The current RAG system provides a robust retrieval foundation, capable of identifying relevant information from the `credit_card_qa.csv` dataset based on semantic similarity.
*   To unlock advanced natural language generation for coherent, synthesized answers, the next crucial step is to integrate a real LLM (e.g., GPT-3.5/4, Llama 2) by configuring `Settings.llm`. This will transform the system from a pure retrieval mechanism into a fully functional RAG system capable of generating sophisticated responses.
